<a href="https://colab.research.google.com/github/xconer-svg/colab-seminar/blob/main/spaceship_titanic_knn_seminar_empty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Семинар 1. EDA и KNN на Spaceship Titanic

На этом семинаре мы решим задачу **бинарной классификации**: по данным о пассажире предскажем, был ли он перенесён в другое измерение (`Transported`).

# Лор
Добро пожаловать в 2912 год, где ваши навыки в области данных науки необходимы для решения космической загадки. Мы получили передачу с четырёх световых лет назад, и ситуация не складывается хорошо.

Космический корабль Титаник был межзвездным пассажирским лайнером, выпущенным месяц назад. С почти 13 000 пассажирами на борту корабль отправился на свой первый рейс, перевозя пассажиров из нашей солнечной системы на три новых приживомых экзопланеты, вращающиеся вокруг близлежащих звезд.

Во время приближения к Альфа-Центаури в пути к своему первому пункту назначения — жаркому 55-канкри Е — неосторожный космический корабль Титаник столкнулся с аномалией пространства и времени, скрытой внутри пыльной облака. К сожалению, ему пришло такое же содействие, как и его названный в честь него предшественник 1000 лет назад. Хотя корабль остался в целостности, почти половина пассажиров была перевезена в альтернативную измерение!

Чтобы помочь спасательным командам и вернуть пропавших пассажиров, вам предстоит предсказать, каких пассажиров было перевездено аномалией, используя записи, восстановленные из поврежденной компьютерной системы космического корабля.

Помогите их спасти и изменить историю!
# Описание файлов и полей данных
### train.csv - личные записи примерно двух третей пассажиров (~8700), используются как обучающие данные.
- `PassengerId` - уникальный идентификатор пассажира. Имеет вид `gggg_pp`, где `gggg` обозначает группу, с которой путешествует пассажир, а `pp` - его номер внутри группы. Люди в одной группе часто оказываются членами семьи, но не всегда.
- `HomePlanet` - планета, с которой отправился пассажир; как правило, это планета его постоянного проживания.
- `CryoSleep` - показывает, выбрал ли пассажир погружение в анабиоз на время полёта. Пассажиры в криосне не покидают свои каюты.
- `Cabin` - номер каюты пассажира. Имеет вид `deck/num/side` (палуба/номер/борт), где `side` принимает значение `P` (Port, левый борт) или `S` (Starboard, правый борт).
- `Destination` - планета, на которой пассажир сойдёт с корабля.
- `Age` - возраст пассажира.
- `VIP` - оплатил ли пассажир специальное VIP-обслуживание на время полёта.
- `RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck` - суммы, которые пассажир потратил в каждом из многочисленных заведений класса люкс на борту Spaceship Titanic.
- `Name` - имя и фамилия пассажира.
- `Transported` - был ли пассажир перенесён в другое измерение. Это целевая переменная: столбец, который нужно предсказать.
### test.csv
- Личные записи оставшейся трети пассажиров (~4300), используются как тестовые данные. Ваша задача - предсказать значение `Transported` для пассажиров из этого набора.
### sample_submission.csv - пример файла с ответами в правильном формате.
- `PassengerId` - идентификатор каждого пассажира из тестового набора.
- `Transported` - целевая переменная. Для каждого пассажира нужно предсказать `True` или `False`.

## 1. Импорт библиотек

Обычно все импорты собирают в начале ноутбука. Сокращения `np`, `pd`, `plt` и `sns` общеприняты.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statistics as stats

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

## 2. Знакомство с NumPy

`numpy` удобен для быстрых вычислений над массивами. Возьмём пример из лекции и посчитаем меры центральной тенденции и разброса.

In [ ]:
# np.array() создаёт массив; от списка он отличается тем, что в массиве лежат данные одного типа
# создайте массив numbers из чисел 4, 8, 6, 5, 3, 100, 7, 6


In [ ]:
# np.unique() возвращает уникальные элементы массива, а с параметром return_counts=True - ещё и сколько раз встретился каждый
# сохраните оба результата в переменные values и counts


In [ ]:
# np.argmax() возвращает индекс наибольшего элемента в массиве, np.argmin() - индекс наименьшего
# найдите моду: значение из values, которое встречается чаще всего, и сохраните её в mode


In [ ]:
# выведите размер массива (shape), среднее (np.mean), медиану (np.median), моду,
# дисперсию (np.var), стандартное отклонение (np.std), размах (max - min) и IQR (через np.percentile)


In [ ]:
# np.where возвращает одно значение, если условие истинно, и другое - если ложно
# пометьте значения больше 10 как "большое", а остальные как "обычное"


Большое значение `100` заметно сдвигает среднее, но гораздо слабее влияет на медиану. Поэтому для скошенных распределений и данных с выбросами пропуски часто заполняют медианой.

## 3. Загружаем и читаем данные с pandas

`pd.read_csv()` превращает CSV-файл в `DataFrame` - таблицу pandas.

На Kaggle данные соревнования уже подключены к ноутбуку. В Colab их нужно загрузить самим: скачайте `train.csv` и `test.csv` со страницы соревнования, нажмите на значок папки слева и перетащите туда оба файла. Ячейка ниже сама найдёт, где они лежат.

In [ ]:
train = pd.read_csv("train.csv", sep = ",",
                    encoding = "utf-8", skipinitialspace = True)
test = pd.read_csv("test.csv", sep = ",",
                   encoding = "utf-8", skipinitialspace = True)

print("Размер train:", train.shape)
print("Размер test: ", test.shape)

Размер train: (8693, 14)
Размер test:  (4277, 13)


In [ ]:
# head() показывает первые строки таблицы
# выведите первые строки train


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


### Что лежит в таблице

- строка - один пассажир;
- признаки (`X`) - сведения о пассажире;
- целевая переменная (`y`) - `Transported`;
- в `test` нет `Transported`: его и должна предсказать модель.

In [ ]:
# выведите список столбцов train (columns)
# и найдите столбцы, которые есть в train, но отсутствуют в test


## 4. Как устроена таблица: DataFrame, Series и индекс

В pandas два главных объекта:

- **`DataFrame`** - таблица целиком: строки и столбцы;
- **`Series`** - один столбец таблицы: набор значений, у каждого из которых есть подпись.

Подписи строк называются **индексом**. В нашей таблице это числа 0, 1, 2, ... - столбец без названия слева в выводе `head()`.

Почти вся работа в pandas выглядит так: взяли из таблицы нужные столбцы, что-то с ними посчитали, собрали результат обратно. Поэтому важно понимать, что у вас сейчас в руках: таблица или столбец.

In [ ]:
# одинарные скобки с названием столбца возвращают Series
# сохраните столбец Age в переменную ages, выведите её тип (type) и первые строки


In [ ]:
# двойные скобки (внутри лежит список названий) возвращают DataFrame, даже если столбец всего один
# выведите тип train[["Age"]] и первые строки таблицы из столбцов Name и Age


Запомнить просто: `train["Age"]` - это столбец, а `train[["Age"]]` - таблица из одного столбца. Разница понадобится уже сегодня: модели из sklearn ждут признаки `X` таблицей, а целевую переменную `y` столбцом.

### Методы Series

У Series есть готовые методы для всего, что мы только что считали в numpy.

In [ ]:
# посчитайте для столбца Age среднее (mean), медиану (median), минимум (min) и максимум (max)
# пропуски pandas при подсчёте просто не учитывает


In [ ]:
# unique() показывает, какие значения встречаются в столбце, nunique() считает, сколько их (пропуски не в счёт)
# примените оба метода к столбцу HomePlanet


In [ ]:
# nunique() работает и для всей таблицы
# посчитайте число уникальных значений в каждом столбце train


У `HomePlanet` всего 3 значения - это категориальный признак. А у `PassengerId` и `Name` значений почти столько же, сколько строк в таблице (8693): в таком виде эти столбцы модели ничего не дадут.

### Индекс

In [ ]:
# выведите индекс таблицы train (index)


Индекс - это подписи строк, а не их номера по порядку. Пока таблицу не трогали, это одно и то же. Но стоит её отсортировать или отфильтровать, и подписи переедут вместе со строками:

In [ ]:
# sort_values() сортирует таблицу по столбцу, ascending=False - по убыванию
# отсортируйте train по возрасту по убыванию, сохраните результат в oldest
# и выведите столбцы Name и Age для первых трёх строк


К различию «подпись или номер по порядку» мы вернёмся в разделе про `loc` и `iloc`.

## 5. Смотрим на данные

In [ ]:
# info() показывает типы столбцов и количество непустых значений


In [ ]:
# describe() считает основные статистики числовых столбцов


посмотрим на количество пропусков в каждой колонке

In [ ]:
# isna() отмечает пропуски, sum() считает их по столбцам


теперь сделаем то же самое, но более витиеватым способом

In [ ]:
# соберите таблицу missing из двух столбцов: missing_count (число пропусков) и missing_percent (их процент)
# sort_values() сортирует датафрейм по указанному столбцу: отсортируйте по missing_count по убыванию


In [ ]:
# посчитайте дубликаты строк и дубликаты PassengerId: duplicated() возвращает маску из True и False
# посмотрите баланс целевой переменной: value_counts() считает, сколько раз встречается каждое значение,
# а с параметром normalize=True переводит всё в доли


Классы сбалансированы: объектов `True` и `False` примерно поровну. Для нас это очень хорошо. Модель не будет склонна предсказывать только определенный класс. Например: у нас сильный дисбаланс: класса 1 всего 5%, а класса 0 95%. Модель может послать класс 1 далеко и надолго и постоянно предсказывать 0, тогда точность будет 95%. Казалось бы, чем нас не устраивает эта метрика. Но в наших интересах предсказывать и класс 1 тоже.

## 6. Выбираем строки и столбцы: loc, iloc и условия

Выбрать часть таблицы можно двумя способами:

- `loc` работает с **подписями**: значениями индекса и названиями столбцов;
- `iloc` работает с **номерами по порядку**, начиная с нуля, как обычные списки Python.

Формат у обоих один: `таблица.loc[строки, столбцы]`.

In [ ]:
# loc: выведите строки с подписями от 0 до 9 и только колонки PassengerId, VIP, Transported


Посчитайте строки: их 10, а не 9. В `loc` правая граница **входит** в результат, потому что это подпись, а не номер. В `iloc` всё как в обычных срезах Python, правая граница не входит:

In [ ]:
# iloc: выведите первые 9 строк и столбцы с номерами 0, 6 и 13 - это те же PassengerId, VIP, Transported


In [ ]:
# iloc: выведите значение из третьей строки и первой колонки таблицы


Теперь вернёмся к таблице `oldest`, которую мы отсортировали по возрасту:

In [ ]:
# выведите возраст из oldest двумя способами: через iloc[0] и через loc[0]


`iloc[0]` вернул возраст самого старшего пассажира: в отсортированной таблице он стоит первым. `loc[0]` нашёл строку с подписью 0, то есть того, кто был первым в исходной таблице. Если нужна «первая по порядку» строка, берите `iloc`.

### Фильтрация по условию

Чаще всего строки выбирают не по номерам, а по условию. Условие над столбцом возвращает Series из `True` и `False`. Её называют **маской**.

In [ ]:
# создайте маску mask: возраст больше 30, и выведите её первые строки


In [ ]:
# передайте условие в loc: выведите пассажиров, чей возраст больше 30


In [ ]:
# True считается как 1, а False как 0
# посчитайте по маске, сколько пассажиров старше 30 (sum) и какова их доля (mean)


Этот приём мы уже видели выше: `train.isna().sum()` - это сумма по маске пропусков.

In [ ]:
# условия можно комбинировать
# выведите пассажиров младше 40, которые потратили больше 5000 в FoodCourt


Условия комбинируют так:

- `(условие 1) & (условие 2)` - выполняются оба («и»);
- `(условие 1) | (условие 2)` - выполняется хотя бы одно («или»);
- `~(условие)` - условие не выполняется («не»).

Каждое условие обязательно берут в скобки. Обычные `and`, `or` и `not` с масками не работают.

In [ ]:
# «или»: выведите Name и Age детей младше 10 лет и пассажиров старше 70
# нужные столбцы передаются в loc вторым аргументом


In [ ]:
# isin() проверяет, входит ли значение в список, а ~ переворачивает маску
# выведите Name и HomePlanet тех, кто летит не с Earth и не с Mars


## 7. Меняем таблицу: новые столбцы и присваивание по условию

Экспериментировать будем на копии, чтобы не испортить исходные данные. `copy()` создаёт независимую таблицу: изменения в `demo` не затронут `train`.

In [ ]:
# создайте копию train в переменной demo
# добавьте в неё столбец IsChild: True, если пассажиру меньше 18 лет
# новый столбец создаётся обычным присваиванием, цикл не нужен


### Присваивание по условию

Чтобы изменить значения только в части строк, нужен уже знакомый `loc`: слева указываем строки и столбец, справа новое значение.

`таблица.loc[условие, "столбец"] = значение`

In [ ]:
# создайте в demo столбец AgeGroup: сначала заполните его значением "взрослый",
# затем через loc поставьте "ребёнок" (младше 18), "пожилой" (от 60) и "неизвестно" (возраст пропущен)
# посмотрите через value_counts(), сколько пассажиров в каждой группе


**Частая ошибка.** Хочется написать короче:

```python
demo[demo["Age"] < 18]["AgeGroup"] = "ребёнок"
```

Так не сработает. Первые скобки создают копию отфильтрованных строк, и новое значение записывается в эту копию, а не в `demo`. pandas покажет предупреждение, а таблица останется прежней. Если видите предупреждение со словами `copy` или `chained assignment`, перепишите строку через `loc`.

### Своя функция: apply

Когда правил много, их удобнее описать обычной функцией Python. `apply()` вызовет её для каждого значения в столбце.

In [ ]:
def age_group(age):
    if pd.isna(age):   # pd.isna() проверяет, что значение - пропуск
        return "неизвестно"
    if age < 18:
        return "ребёнок"
    if age >= 60:
        return "пожилой"
    return "взрослый"

# примените её к столбцу Age через apply() и проверьте, что результат совпал со столбцом AgeGroup


### Замена значений и работа с текстом

In [ ]:
# map() заменяет значения по словарю: что было -> что стало
# создайте столбец HomePlanetRu с русскими названиями планет


In [ ]:
# у текстовых столбцов есть набор методов .str
# разделите Cabin по символу "/" через str.split; expand=True разложит части по отдельным столбцам


В `Cabin` спрятаны сразу три признака: палуба, номер каюты и сторона корабля. В разделе про Feature Engineering мы вытащим их в отдельные столбцы именно так.

## 8. Группировка: groupby, agg и crosstab

Самые полезные вопросы к данным звучат как «посчитай что-то **для каждой** группы»: средний возраст для каждой планеты, доля перенесённых для каждой палубы. За это отвечает `groupby`. Он делает три шага:

1. **разбивает** строки на группы по значению столбца;
2. **считает** внутри каждой группы то, что мы попросили;
3. **собирает** результаты в одну таблицу.

Запись читается слева направо. `train.groupby("HomePlanet")["Age"].mean()` - «сгруппируй по планете, возьми возраст, посчитай среднее».

In [ ]:
# посчитайте средний возраст для каждой HomePlanet


Получилась Series: в индексе стоят названия групп, в значениях результат для каждой.

In [ ]:
# Transported состоит из True и False, поэтому среднее по нему - это доля перенесённых
# посчитайте эту долю для каждой HomePlanet


С Europa перенесло 66% пассажиров, а с Earth только 42%. Значит, `HomePlanet` - полезный признак для модели. Так `groupby` помогает находить закономерности ещё до обучения.

In [ ]:
# agg() считает сразу несколько статистик
# для каждой HomePlanet найдите среднее, медиану и число непустых значений (count) возраста


In [ ]:
# группировать можно сразу по нескольким столбцам
# посчитайте долю перенесённых для каждого сочетания HomePlanet и CryoSleep


Пассажиров с Europa, которые летели в криосне, перенесло почти всех: 99%.

По умолчанию `groupby` пропускает строки, у которых в столбце группировки стоит пропуск. Чтобы увидеть их отдельной группой, добавьте параметр `dropna=False`.

### crosstab: таблица «признак на признак»

Когда нужно посчитать, сколько строк приходится на каждое сочетание двух категорий, удобнее `pd.crosstab`.

In [ ]:
# pd.crosstab: строки - HomePlanet, столбцы - Transported, в ячейках число пассажиров


In [ ]:
# постройте ту же таблицу в долях: normalize="index" считает доли внутри каждой строки


## 9. Data Analysis


In [ ]:
# постройте распределение возраста в seaborn: sns.histplot с параметрами kde=True и stat="density"


In [ ]:
# постройте homeplanet_rate столбчатой диаграммой в seaborn: sns.barplot


### Расходы пассажиров

Создадим новый признак: общую сумму расходов по пяти сервисам корабля. `sum(axis=1)` складывает значения по строке.

In [ ]:
# сохраните названия пяти столбцов с тратами в список spend_columns
# создайте копию train в переменной train_eda и добавьте в неё столбец TotalSpend: сумму трат по строке (sum(axis=1))
# такой же столбец добавьте в test


In [ ]:
# добавьте в train_eda и в test столбец is_spends: 1, если пассажир что-то потратил, и 0, если нет


мы были не совсем правы
всё-таки, есть випы, которые ничего не потратили

## 10. Тепловая карта корреляций в seaborn

In [ ]:
# корреляция измеряет линейную связь числовых признаков
# выберите числовые столбцы train_eda (select_dtypes) в таблицу numeric_for_corr
# и добавьте в неё Transported, превратив True и False в 1 и 0 через astype(int)


In [ ]:
# посчитайте корреляции (corr) и постройте тепловую карту: sns.heatmap с параметром annot=True


Важно: корреляция не показывает причинно-следственную связь, также мы ничего не можем сказать про зависимость в целом. Корреляция может сообщить нам только о линейной зависимости признаков. Если признаки зависят, допустим, по косинусу, то корреляция нам об этом ничего не скажет. Кроме того, карта не показывает категориальные признаки, пока мы не перевели их в числа.

## 11. Preprocessing и Feature Engineering

KNN сравнивает объекты по расстояниям, поэтому:

1. пропуски нужно заполнить;
2. категориальные признаки нужно закодировать числами;
3. числовые признаки нужно привести к сопоставимому масштабу, то есть нормализовать.

Из `Cabin` вида `deck/num/side` возьмём палубу и сторону.

Из `PassengerId` вида `gggg_pp` возьмём группу.

In [ ]:
# напишите функцию make_features(df), которая возвращает копию таблицы с новыми признаками:
# 1. TotalSpend - сумма трат по пяти сервисам
# 2. CabinDeck и CabinSide - палуба и сторона из Cabin
# 3. PassengerGroup - номер группы из PassengerId
# 4. столбцы PassengerId, Cabin и Name удаляются (drop)
# примените её: X - признаки train_eda без Transported, y - Transported в виде 0 и 1, X_test - признаки test


In [ ]:
# разделите названия столбцов X на два списка: numeric_features (числовые) и categorical_features (все остальные)


## 12. Разделяем данные на обучение и валидацию

In [ ]:
# разделите X и y на обучающую и валидационную части через train_test_split:
# test_size=0.2, random_state=RANDOM_STATE, stratify=y
# результат сохраните в X_train, X_valid, y_train, y_valid
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train.shape
X_valid.shape

NameError: name 'train_test_split' is not defined

`stratify=y` сохраняет примерно одинаковую долю классов в обеих частях. Валидацию модель не видит во время обучения - на ней мы честно оцениваем качество и выбираем `k`.

## 13. Preprocessing и KNN в Pipeline

Pipeline - программный конвейер, который объединяет несколько операций в одну общую цепочку

SimpleImputer() помогает заполнять нам пропуски( strategy = mean/median/most_frequent/constant задается отдельно параметром fill_value)

OneHotEncoder() - энкодер, который помогает преобразовывать категориальные признаки в числовые(показать пример на доске)

In [ ]:
# соберите предобработку и модель в один Pipeline:
# 1. numeric_transformer: заполнение пропусков медианой (SimpleImputer) и StandardScaler
# 2. categorical_transformer: заполнение самым частым значением и OneHotEncoder
# 3. preprocessor: ColumnTransformer, который применяет их к числовым и категориальным столбцам
# 4. knn_pipeline: preprocessor и KNeighborsClassifier


`Pipeline` важен не только для удобства. Все медианы, моды и параметры масштабирования вычисляются **только на обучающей части**, поэтому информация из валидации не подглядывается.

## 14. Выбираем число соседей k

In [ ]:
# переберите k от 3 до 25 с шагом 2: для каждого k обучите knn_pipeline на обучающей части
# и посчитайте accuracy на трейне и на валидации
# результаты соберите в таблицу scores со столбцами k, validation_accuracy и train_accuracy


что мы затронем в программе курса

In [ ]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(max_depth = 5, eta = 4e-3, n_estimators = 1000)),
])

In [ ]:
xgb_pipeline.fit(X_train, y_train)
preds = xgb_pipeline.predict(X_valid)
score = accuracy_score(preds, y_valid)
score

KeyboardInterrupt: 

ничего не выбирали, не тюнили, параметры, проставленные просто так, дали скор лучше, чем knn после перебора k соседей

## 15. Оцениваем выбранную модель

Accuracy: 0.797


## 16. Обучаем финальную KNN и создаём submission.csv

## Итоги

Мы прошли полный первый ML-пайплайн:

1. загрузили и исследовали данные;
2. нашли пропуски и посмотрели распределения;
3. создали простые признаки;
4. разделили данные на train и validation;
5. заполнили пропуски, закодировали категории и масштабировали числа;
6. подобрали гиперпараметр `k`;
7. оценили KNN;
8. обучили финальную модель и сохранили предсказания.

## Шпаргалка по pandas

| Что нужно сделать | Как это записать |
|---|---|
| Прочитать файл | `pd.read_csv("file.csv")` |
| Быстро осмотреть таблицу | `df.head()`, `df.shape`, `df.info()`, `df.describe()` |
| Взять столбец или несколько | `df["a"]` - Series, `df[["a", "b"]]` - DataFrame |
| Посчитать статистику по столбцу | `.mean()`, `.median()`, `.min()`, `.max()`, `.sum()` |
| Узнать, какие значения встречаются | `.unique()`, `.nunique()`, `.value_counts()` |
| Найти и заполнить пропуски | `df.isna().sum()`, `.fillna(значение)`, `.dropna()` |
| Найти дубликаты | `df.duplicated().sum()` |
| Выбрать по подписям или по номерам | `df.loc[строки, столбцы]`, `df.iloc[строки, столбцы]` |
| Выбрать строки по условию | `df.loc[(df["a"] > 1) & (df["b"] == "x")]`, `.isin([...])` |
| Отсортировать | `df.sort_values("a", ascending=False)` |
| Создать столбец | `df["c"] = df["a"] + df["b"]` |
| Изменить значения по условию | `df.loc[условие, "a"] = значение` |
| Заменить значения, применить функцию | `.map(словарь)`, `.apply(функция)` |
| Разобрать текст | `.str.split("/", expand=True)` |
| Сменить тип | `.astype(int)` |
| Посчитать по группам | `df.groupby("a")["b"].mean()`, `.agg(["mean", "max"])` |
| Таблица сочетаний двух признаков | `pd.crosstab(df["a"], df["b"])` |
| Удалить столбцы | `df.drop(columns=["a", "b"])` |
| Сохранить в файл | `df.to_csv("file.csv", index=False)` |
